# HCFIM EEG Bootstrap Confidence Intervals V1

This notebook computes nonparametric confidence intervals for the classifier metrics reported in the HCFIM EEG paper.

The notebook does **not** retrain models. It uses the subject-level out-of-fold predictions generated by:

- `01_hcfim_eeg_paper_reproducible.ipynb`
- `02_hcfim_eeg_lateralized_v1_analysis.ipynb`

The bootstrap unit is the subject. Epochs are not resampled and are not treated as independent observations.


## tl;dr

Execute the full notebook to generate:

- `results/classification_metric_bootstrap_ci_long.csv`
- `results/classification_results_with_bootstrap_ci.csv`
- `results/classification_results_lateralized_v1_with_bootstrap_ci.csv`
- `results/paper_candidate_models_with_bootstrap_ci.csv`
- `figures/classification_candidate_auc_bootstrap_ci.png`
- `figures/classification_candidate_balanced_accuracy_bootstrap_ci.png`

Interpretation rule: report the observed out-of-fold metric with a percentile bootstrap 95% confidence interval.


## 1. Context and Assumptions

Bootstrap confidence intervals are computed from already-generated out-of-fold predictions. This is appropriate here because the paper reports subject-level cross-validated performance, and reviewers are requesting uncertainty intervals around those estimates.

Key assumptions:

- Resampling unit: subject.
- Bootstrap type: stratified nonparametric bootstrap.
- Class balance within each diagnostic comparison is preserved in each replicate.
- ROC AUC is recomputed from `y_prob`.
- Confusion-matrix metrics are recomputed from the fixed out-of-fold `y_pred` column.
- No model fitting, feature selection or scaling is performed in this notebook.

This CI estimates uncertainty in the out-of-fold prediction summary. It does not replace an external validation cohort.


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import math
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

def resolve_project_root() -> Path:
    """Locate the repository without relying on a machine-specific path."""
    env_root = os.environ.get("HCFIM_PROJECT_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / "notebooks").is_dir() and (candidate / "src").is_dir():
            return candidate
        raise FileNotFoundError(
            "HCFIM_PROJECT_ROOT does not point to the HCFIM-eeg-research repository."
        )

    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository. Run the notebook from within the repository "
        "or set HCFIM_PROJECT_ROOT."
    )

PROJECT_ROOT = resolve_project_root()
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

N_BOOTSTRAP = 5000
CI_LEVEL = 0.95
BOOTSTRAP_RANDOM_STATE = 17536

alpha = 1.0 - CI_LEVEL
CI_LOW_Q = 100 * alpha / 2.0
CI_HIGH_Q = 100 * (1.0 - alpha / 2.0)

print("Project root:", PROJECT_ROOT)
print("Bootstrap replicates:", N_BOOTSTRAP)
print("CI level:", CI_LEVEL)


## 2. Load Out-of-Fold Predictions


In [ ]:
def load_oof_predictions(results_dir: Path) -> pd.DataFrame:
    paths = sorted(results_dir.glob("oof_predictions*.csv"))
    if not paths:
        raise FileNotFoundError(f"No out-of-fold prediction files found in {results_dir}")

    frames = []
    for path in paths:
        df = pd.read_csv(path)
        required = {"participant_id", "diagnosis", "y_true", "y_prob", "y_pred", "comparison", "feature_family", "model"}
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"{path.name} is missing required columns: {sorted(missing)}")
        if "schema" not in df.columns:
            df["schema"] = "regional_v1"
        df["source_file"] = path.name
        frames.append(df)

    out = pd.concat(frames, ignore_index=True)
    out["y_true"] = out["y_true"].astype(int)
    out["y_pred"] = out["y_pred"].astype(int)
    out["y_prob"] = out["y_prob"].astype(float)
    return out


oof = load_oof_predictions(RESULTS_DIR)
print("OOF shape:", oof.shape)
display(
    oof.groupby(["schema", "comparison", "feature_family", "model"], as_index=False)
    .agg(n_subjects=("participant_id", "nunique"), rows=("participant_id", "size"))
    .sort_values(["schema", "comparison", "feature_family", "model"])
    .head(12)
)

## 3. Metric and Bootstrap Functions


In [ ]:
METRICS = ["roc_auc", "balanced_accuracy", "sensitivity", "specificity", "f1"]


def fast_auc(pos_prob: np.ndarray, neg_prob: np.ndarray) -> float:
    pos_prob = np.asarray(pos_prob, dtype=float)
    neg_prob = np.asarray(neg_prob, dtype=float)
    if len(pos_prob) == 0 or len(neg_prob) == 0:
        return np.nan
    comparisons = pos_prob[:, None] - neg_prob[None, :]
    return float((comparisons > 0).mean() + 0.5 * (comparisons == 0).mean())


def compute_binary_metrics(y_true: np.ndarray, y_prob: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = np.asarray(y_pred, dtype=int)

    pos_mask = y_true == 1
    neg_mask = y_true == 0
    tp = int(np.sum(y_pred[pos_mask] == 1))
    fn = int(np.sum(y_pred[pos_mask] == 0))
    tn = int(np.sum(y_pred[neg_mask] == 0))
    fp = int(np.sum(y_pred[neg_mask] == 1))

    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    balanced_accuracy = np.nanmean([sensitivity, specificity])
    f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0

    roc_auc = fast_auc(y_prob[pos_mask], y_prob[neg_mask])

    return {
        "roc_auc": roc_auc,
        "balanced_accuracy": balanced_accuracy,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "f1": f1,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def stratified_bootstrap_metric_ci(
    predictions: pd.DataFrame,
    n_bootstrap: int = N_BOOTSTRAP,
    random_state: int = BOOTSTRAP_RANDOM_STATE,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(random_state)
    predictions = predictions.reset_index(drop=True)

    class_indices = {
        cls: predictions.index[predictions["y_true"] == cls].to_numpy()
        for cls in sorted(predictions["y_true"].unique())
    }
    if set(class_indices) != {0, 1}:
        raise ValueError("Each model must contain both classes for stratified bootstrap.")

    neg = predictions.loc[class_indices[0]]
    pos = predictions.loc[class_indices[1]]
    n_neg = len(neg)
    n_pos = len(pos)

    neg_prob = neg["y_prob"].to_numpy(dtype=float)
    pos_prob = pos["y_prob"].to_numpy(dtype=float)
    neg_pred = neg["y_pred"].to_numpy(dtype=int)
    pos_pred = pos["y_pred"].to_numpy(dtype=int)

    observed = compute_binary_metrics(
        predictions["y_true"].to_numpy(),
        predictions["y_prob"].to_numpy(),
        predictions["y_pred"].to_numpy(),
    )

    pos_sample_idx = rng.integers(0, n_pos, size=(n_bootstrap, n_pos))
    neg_sample_idx = rng.integers(0, n_neg, size=(n_bootstrap, n_neg))

    pos_prob_samples = pos_prob[pos_sample_idx]
    neg_prob_samples = neg_prob[neg_sample_idx]
    pairwise = pos_prob_samples[:, :, None] - neg_prob_samples[:, None, :]
    roc_auc_values = (pairwise > 0).mean(axis=(1, 2)) + 0.5 * (pairwise == 0).mean(axis=(1, 2))

    pos_pred_samples = pos_pred[pos_sample_idx]
    neg_pred_samples = neg_pred[neg_sample_idx]

    tp = pos_pred_samples.sum(axis=1).astype(float)
    fn = n_pos - tp
    fp = neg_pred_samples.sum(axis=1).astype(float)
    tn = n_neg - fp

    sensitivity_values = tp / n_pos
    specificity_values = tn / n_neg
    balanced_accuracy_values = 0.5 * (sensitivity_values + specificity_values)
    f1_denominator = 2 * tp + fp + fn
    f1_values = np.divide(
        2 * tp,
        f1_denominator,
        out=np.zeros_like(tp, dtype=float),
        where=f1_denominator != 0,
    )

    samples = {
        "roc_auc": roc_auc_values,
        "balanced_accuracy": balanced_accuracy_values,
        "sensitivity": sensitivity_values,
        "specificity": specificity_values,
        "f1": f1_values,
    }

    summary_rows = []
    for metric in METRICS:
        values = samples[metric]
        summary_rows.append({
            "metric": metric,
            "observed": observed[metric],
            "bootstrap_mean": np.nanmean(values),
            "bootstrap_std": np.nanstd(values, ddof=1),
            "ci_low": np.nanpercentile(values, CI_LOW_Q),
            "ci_high": np.nanpercentile(values, CI_HIGH_Q),
            "ci_level": CI_LEVEL,
            "n_bootstrap": n_bootstrap,
            "ci_method": "stratified_subject_percentile_bootstrap",
        })

    return pd.DataFrame(summary_rows)


## 4. Run Bootstrap for All Models


In [ ]:
grouping_columns = ["schema", "comparison", "feature_family", "model"]
ci_frames = []

for group_key, group_df in oof.groupby(grouping_columns, sort=True):
    schema, comparison, feature_family, model = group_key
    ci_summary = stratified_bootstrap_metric_ci(group_df)
    for column, value in zip(grouping_columns, group_key):
        ci_summary[column] = value
    ci_summary["n_subjects"] = group_df["participant_id"].nunique()
    ci_summary["n_controls"] = int((group_df["y_true"] == 0).sum())
    ci_summary["n_cases"] = int((group_df["y_true"] == 1).sum())
    ci_frames.append(ci_summary)

metric_ci_long = pd.concat(ci_frames, ignore_index=True)
metric_ci_long = metric_ci_long[
    grouping_columns
    + ["n_subjects", "n_controls", "n_cases", "metric", "observed", "bootstrap_mean", "bootstrap_std", "ci_low", "ci_high", "ci_level", "n_bootstrap", "ci_method"]
].sort_values(grouping_columns + ["metric"])

metric_ci_long.to_csv(RESULTS_DIR / "classification_metric_bootstrap_ci_long.csv", index=False)
print("Saved:", RESULTS_DIR / "classification_metric_bootstrap_ci_long.csv")
print("Rows:", len(metric_ci_long))
display(metric_ci_long.head(12))


## 5. Merge Confidence Intervals With Classification Tables


In [ ]:
def make_wide_ci(metric_ci: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for keys, g in metric_ci.groupby(grouping_columns, sort=True):
        row = dict(zip(grouping_columns, keys))
        for _, mrow in g.iterrows():
            metric = mrow["metric"]
            row[f"{metric}_observed"] = mrow["observed"]
            row[f"{metric}_ci_low"] = mrow["ci_low"]
            row[f"{metric}_ci_high"] = mrow["ci_high"]
            row[f"{metric}_bootstrap_mean"] = mrow["bootstrap_mean"]
            row[f"{metric}_bootstrap_std"] = mrow["bootstrap_std"]
        rows.append(row)
    return pd.DataFrame(rows)


wide_ci = make_wide_ci(metric_ci_long)

regional_metrics = pd.read_csv(RESULTS_DIR / "classification_results.csv")
regional_metrics["schema"] = "regional_v1"
lateralized_metrics = pd.read_csv(RESULTS_DIR / "classification_results_lateralized_v1.csv")

regional_with_ci = regional_metrics.merge(wide_ci, on=grouping_columns, how="left")
lateralized_with_ci = lateralized_metrics.merge(wide_ci, on=grouping_columns, how="left")

regional_with_ci.to_csv(RESULTS_DIR / "classification_results_with_bootstrap_ci.csv", index=False)
lateralized_with_ci.to_csv(RESULTS_DIR / "classification_results_lateralized_v1_with_bootstrap_ci.csv", index=False)

all_with_ci = pd.concat([regional_with_ci, lateralized_with_ci], ignore_index=True, sort=False)
all_with_ci.to_csv(RESULTS_DIR / "classification_results_all_schemas_with_bootstrap_ci.csv", index=False)

print("Saved:")
print(" -", RESULTS_DIR / "classification_results_with_bootstrap_ci.csv")
print(" -", RESULTS_DIR / "classification_results_lateralized_v1_with_bootstrap_ci.csv")
print(" -", RESULTS_DIR / "classification_results_all_schemas_with_bootstrap_ci.csv")

display(all_with_ci.sort_values(["comparison", "roc_auc"], ascending=[True, False]).head(10))


## 6. Paper Candidate Models


In [ ]:
candidate_specs = [
    {
        "paper_role": "AD regional primary",
        "schema": "regional_v1",
        "comparison": "control_vs_alzheimer",
        "feature_family": "psd_plus_hcfim",
        "model": "extra_trees",
    },
    {
        "paper_role": "AD lateralized sensitivity extension",
        "schema": "lateralized_v1",
        "comparison": "control_vs_alzheimer",
        "feature_family": "psd_plus_hcfim_plus_lateral_deltas",
        "model": "extra_trees",
    },
    {
        "paper_role": "FTD regional best ROC AUC",
        "schema": "regional_v1",
        "comparison": "control_vs_frontotemporal_dementia",
        "feature_family": "hcfim",
        "model": "mlp_small",
    },
    {
        "paper_role": "FTD regional best balanced accuracy",
        "schema": "regional_v1",
        "comparison": "control_vs_frontotemporal_dementia",
        "feature_family": "psd_plus_hcfim",
        "model": "extra_trees",
    },
    {
        "paper_role": "FTD lateralized exploratory",
        "schema": "lateralized_v1",
        "comparison": "control_vs_frontotemporal_dementia",
        "feature_family": "psd_plus_hcfim",
        "model": "logistic_elasticnet",
    },
]

candidate_rows = []
for spec in candidate_specs:
    mask = np.logical_and.reduce([
        all_with_ci[key] == value
        for key, value in spec.items()
        if key != "paper_role"
    ])
    selected = all_with_ci.loc[mask].copy()
    if selected.empty:
        raise ValueError(f"Candidate not found: {spec}")
    selected.insert(0, "paper_role", spec["paper_role"])
    candidate_rows.append(selected)

paper_candidates = pd.concat(candidate_rows, ignore_index=True)
paper_candidates.to_csv(RESULTS_DIR / "paper_candidate_models_with_bootstrap_ci.csv", index=False)

display_columns = [
    "paper_role", "schema", "comparison", "feature_family", "model",
    "n_subjects", "roc_auc", "roc_auc_ci_low", "roc_auc_ci_high",
    "balanced_accuracy", "balanced_accuracy_ci_low", "balanced_accuracy_ci_high",
    "sensitivity", "sensitivity_ci_low", "sensitivity_ci_high",
    "specificity", "specificity_ci_low", "specificity_ci_high",
]
display(paper_candidates[display_columns])
print("Saved:", RESULTS_DIR / "paper_candidate_models_with_bootstrap_ci.csv")


## 7. Figures for Reviewer-Facing Results


In [ ]:
def plot_candidate_metric_ci(candidates: pd.DataFrame, metric: str, output_path: Path) -> None:
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("matplotlib is not available; skipping figure:", output_path.name)
        return

    plot_df = candidates.copy()
    labels = plot_df["paper_role"].to_list()
    y = plot_df[metric].to_numpy(dtype=float)
    low = plot_df[f"{metric}_ci_low"].to_numpy(dtype=float)
    high = plot_df[f"{metric}_ci_high"].to_numpy(dtype=float)
    x = np.arange(len(plot_df))

    plt.rcParams["font.family"] = "Arial"
    fig, ax = plt.subplots(figsize=(11, 5.8))
    ax.errorbar(
        x,
        y,
        yerr=np.vstack([y - low, high - y]),
        fmt="o",
        color="#2F5D8C",
        ecolor="#6E7F91",
        elinewidth=2,
        capsize=5,
        markersize=7,
    )
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=25, ha="right")
    ax.set_ylabel(metric.replace("_", " ").title())
    ax.set_ylim(0.35, 1.0)
    ax.grid(axis="y", alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    fig.tight_layout()
    fig.savefig(output_path, dpi=220, bbox_inches="tight")
    plt.show()
    print("Saved:", output_path)


plot_candidate_metric_ci(
    paper_candidates,
    "roc_auc",
    FIGURES_DIR / "classification_candidate_auc_bootstrap_ci.png",
)
plot_candidate_metric_ci(
    paper_candidates,
    "balanced_accuracy",
    FIGURES_DIR / "classification_candidate_balanced_accuracy_bootstrap_ci.png",
)


## 8. Best Models With Confidence Intervals


In [ ]:
best_by_auc = (
    all_with_ci
    .sort_values(["schema", "comparison", "roc_auc"], ascending=[True, True, False])
    .groupby(["schema", "comparison"], as_index=False)
    .head(1)
    .assign(selection_rule="best_roc_auc")
)

best_by_balanced = (
    all_with_ci
    .sort_values(["schema", "comparison", "balanced_accuracy"], ascending=[True, True, False])
    .groupby(["schema", "comparison"], as_index=False)
    .head(1)
    .assign(selection_rule="best_balanced_accuracy")
)

best_models_with_ci = pd.concat([best_by_auc, best_by_balanced], ignore_index=True)
best_models_with_ci.to_csv(RESULTS_DIR / "best_models_with_bootstrap_ci.csv", index=False)

cols = [
    "selection_rule", "schema", "comparison", "feature_family", "model",
    "roc_auc", "roc_auc_ci_low", "roc_auc_ci_high",
    "balanced_accuracy", "balanced_accuracy_ci_low", "balanced_accuracy_ci_high",
    "sensitivity", "sensitivity_ci_low", "sensitivity_ci_high",
    "specificity", "specificity_ci_low", "specificity_ci_high",
]
display(best_models_with_ci[cols])
print("Saved:", RESULTS_DIR / "best_models_with_bootstrap_ci.csv")


## 9. Reporting Template

Recommended wording:

> Performance was estimated using leakage-aware subject-level out-of-fold predictions. Uncertainty was quantified using a stratified nonparametric bootstrap over subjects with 5,000 replicates. For each replicate, subjects were sampled with replacement within each class, preserving the original number of controls and cases for the binary comparison. Percentile 95% confidence intervals were computed for ROC AUC, balanced accuracy, sensitivity, specificity and F1 score.

Caveat:

> These intervals quantify uncertainty of the out-of-fold performance estimate in the available public cohort. They do not replace an external validation cohort.
